# 🌫️ Diffusion Models

This notebook builds deep intuition for:
1. **The forward process** — systematically destroying information with noise
2. **The reverse process** — learning to denoise step by step
3. **The noise schedule** — how fast to add noise
4. **Score matching** — learning the gradient of log-probability
5. **DDPM** — the denoising diffusion probabilistic model
6. **The reparameterization trick** — predicting noise instead of data
7. **Sampling** — generating data by iterative denoising
8. **Connection to score-based models** — two perspectives, one framework

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 1. Destroying and Reconstructing Information

Imagine you have a beautiful photograph. You photocopy it, adding a tiny bit of noise.
Then photocopy the photocopy. Repeat 1000 times. After enough copies, you have **pure static**.

Now imagine you could **reverse** this process — take static and gradually reconstruct the image.

**Why this works:**
- Destroying information is trivial (just add noise)
- Each tiny destruction step has a nearly Gaussian reverse
- If you learn to undo each small step, you can create from nothing

Let's watch information being destroyed.

In [ ]:
# The forward process: watching data dissolve into noise

# Create interesting 2D data
def make_moons(n, noise=0.06):
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.column_stack([np.cos(t), np.sin(t)])
    x2 = np.column_stack([1 - np.cos(t), -np.sin(t) + 0.5])
    return np.vstack([x1, x2]) + np.random.randn(n, 2) * noise

data = make_moons(3000)

# Noise schedule
T = 500
beta = np.linspace(1e-4, 0.02, T)
alpha = 1 - beta
alpha_bar = np.cumprod(alpha)

# Forward process: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1-alpha_bar_t) * epsilon
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
timesteps = [0, 25, 75, 150, 250, 350, 425, 475, 499, -1]

for i, t in enumerate(timesteps):
    row, col = i // 5, i % 5
    ax = axes[row, col]
    
    if t == 0:
        xt = data
        snr = float('inf')
    elif t == -1:
        xt = np.random.randn(len(data), 2)  # pure noise
        snr = 0
    else:
        epsilon = np.random.randn(len(data), 2)
        xt = np.sqrt(alpha_bar[t]) * data + np.sqrt(1 - alpha_bar[t]) * epsilon
        snr = alpha_bar[t] / (1 - alpha_bar[t])
    
    ax.scatter(xt[:, 0], xt[:, 1], s=1, alpha=0.3, c='steelblue')
    t_label = 'Pure noise' if t == -1 else f't={t}'
    snr_label = '∞' if t == 0 else ('0' if t == -1 else f'{snr:.2f}')
    ax.set_title(f'{t_label}\nSNR={snr_label}', fontsize=10)
    ax.set_xlim(-4, 5)
    ax.set_ylim(-3, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.suptitle('Forward Diffusion: Data → Noise (Information Destruction)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("🔑 Key Insight: We SYSTEMATICALLY destroy information by adding noise.")
print("The forward process requires NO learning — it's just adding Gaussian noise!")
print("The magic is in learning to REVERSE this process.")

## 2. The Forward Process — Mathematics

At each step: $q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} x_{t-1}, \beta_t I)$

The beautiful trick: we can **jump directly** to any timestep!

$$x_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1 - \bar{\alpha}_t} \, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

where $\bar{\alpha}_t = \prod_{s=1}^t (1 - \beta_s)$

- $\sqrt{\bar{\alpha}_t}$ = **signal strength** (decreases from 1 to ~0)
- $\sqrt{1-\bar{\alpha}_t}$ = **noise strength** (increases from ~0 to 1)

In [ ]:
# Signal vs Noise over time

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Linear schedule
axes[0].plot(np.sqrt(alpha_bar), 'steelblue', linewidth=2, label='Signal √ᾱₜ')
axes[0].plot(np.sqrt(1 - alpha_bar), 'coral', linewidth=2, label='Noise √(1-ᾱₜ)')
axes[0].fill_between(range(T), 0, np.sqrt(alpha_bar), alpha=0.1, color='steelblue')
axes[0].fill_between(range(T), 0, np.sqrt(1-alpha_bar), alpha=0.1, color='coral')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('Coefficient')
axes[0].set_title('Signal and Noise Strengths')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# SNR
snr = alpha_bar / (1 - alpha_bar + 1e-10)
axes[1].semilogy(snr, 'green', linewidth=2)
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('SNR (log scale)')
axes[1].set_title('Signal-to-Noise Ratio')
axes[1].grid(True, alpha=0.3)

# 1D example: one data point being noised
x0_1d = 3.0  # a data point
n_trajectories = 20
t_range = np.arange(T)

for _ in range(n_trajectories):
    epsilon = np.random.randn(T)
    xt = np.sqrt(alpha_bar) * x0_1d + np.sqrt(1 - alpha_bar) * epsilon
    axes[2].plot(t_range, xt, 'steelblue', alpha=0.2, linewidth=0.5)

axes[2].axhline(y=x0_1d, color='red', linestyle='--', linewidth=2, label='Original x₀')
axes[2].axhline(y=0, color='gray', linestyle=':', alpha=0.5)
axes[2].set_xlabel('Timestep t')
axes[2].set_ylabel('x_t')
axes[2].set_title('1D Trajectories: Data → Noise')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At t=0: signal={np.sqrt(alpha_bar[0]):.4f}, noise={np.sqrt(1-alpha_bar[0]):.4f}")
print(f"At t={T//2}: signal={np.sqrt(alpha_bar[T//2]):.4f}, noise={np.sqrt(1-alpha_bar[T//2]):.4f}")
print(f"At t={T-1}: signal={np.sqrt(alpha_bar[-1]):.4f}, noise={np.sqrt(1-alpha_bar[-1]):.4f}")
print("\nThe signal fades while noise grows — information is gradually destroyed.")

## 3. The Noise Schedule — How Fast to Destroy

The noise schedule $\{\beta_t\}$ controls how quickly information is destroyed.

Two popular choices:
- **Linear:** $\beta_t$ increases linearly. Simple but wastes early timesteps.
- **Cosine:** More uniform SNR decrease. Better utilization of all timesteps.

In [ ]:
# Compare linear vs cosine schedules

# Cosine schedule
s = 0.008
t_cos = np.arange(T + 1)
f_cos = np.cos((t_cos/T + s) / (1+s) * np.pi/2)**2
alpha_bar_cosine = f_cos[1:] / f_cos[0]
alpha_bar_cosine = np.clip(alpha_bar_cosine, 1e-5, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Alpha_bar comparison
axes[0].plot(alpha_bar, 'b-', linewidth=2, label='Linear')
axes[0].plot(alpha_bar_cosine, 'r-', linewidth=2, label='Cosine')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('ᾱₜ')
axes[0].set_title('ᾱₜ: How Much Signal Remains')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# SNR comparison
snr_linear = alpha_bar / (1 - alpha_bar + 1e-10)
snr_cosine = alpha_bar_cosine / (1 - alpha_bar_cosine + 1e-10)
axes[1].semilogy(snr_linear, 'b-', linewidth=2, label='Linear')
axes[1].semilogy(snr_cosine, 'r-', linewidth=2, label='Cosine')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('SNR (log scale)')
axes[1].set_title('SNR Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Noised data comparison at t=T/2
t_mid = T // 2
eps = np.random.randn(len(data), 2)
xt_lin = np.sqrt(alpha_bar[t_mid]) * data + np.sqrt(1-alpha_bar[t_mid]) * eps
xt_cos = np.sqrt(alpha_bar_cosine[t_mid]) * data + np.sqrt(1-alpha_bar_cosine[t_mid]) * eps

axes[2].scatter(xt_lin[:500, 0], xt_lin[:500, 1], s=3, alpha=0.3, c='steelblue', label='Linear')
axes[2].scatter(xt_cos[:500, 0], xt_cos[:500, 1], s=3, alpha=0.3, c='coral', label='Cosine')
axes[2].set_title(f'Noised Data at t={t_mid}\nLinear: more noisy, Cosine: less noisy')
axes[2].legend()
axes[2].set_aspect('equal')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🔑 The cosine schedule destroys information more gradually.")
print("This means the model can learn from ALL timesteps, not just a narrow band.")

## 4. The DDPM Training Objective

The loss is beautifully simple — just **predict the noise that was added**:

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

Training algorithm:
1. Sample $x_0$ from data
2. Sample $t \sim \text{Uniform}\{1, ..., T\}$
3. Sample $\epsilon \sim \mathcal{N}(0, I)$
4. Compute $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$
5. Minimize $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$

**This is just denoising at different noise levels!**

In [ ]:
# DDPM Training Implementation

np.random.seed(42)

# Sinusoidal time embedding
def time_embed(t, dim=16):
    freqs = np.exp(np.linspace(0, -4, dim // 2))
    args = t[:, None] * freqs[None, :]
    return np.column_stack([np.sin(args), np.cos(args)])

# Noise predictor MLP
class NoiseNet:
    def __init__(self, t_dim=16, hidden=48):
        self.t_dim = t_dim
        inp = 2 + t_dim
        self.W1 = np.random.randn(inp, hidden) * 0.2
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, hidden) * 0.2
        self.b2 = np.zeros(hidden)
        self.W3 = np.random.randn(hidden, 2) * 0.05
        self.b3 = np.zeros(2)
    
    def predict(self, xt, t_idx):
        t_emb = time_embed(t_idx.astype(float) / T, self.t_dim)
        inp = np.column_stack([xt, t_emb])
        h = np.tanh(inp @ self.W1 + self.b1)
        h = np.tanh(h @ self.W2 + self.b2)
        return h @ self.W3 + self.b3
    
    def params(self):
        return [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3]

model = NoiseNet(t_dim=16, hidden=48)

# Training
lr = 0.002
batch_size = 256
losses = []

for step in range(3000):
    idx = np.random.randint(0, len(data), batch_size)
    x0 = data[idx]
    t = np.random.randint(0, T, batch_size)
    eps = np.random.randn(batch_size, 2)
    
    # Forward: noise the data
    xt = np.sqrt(alpha_bar[t])[:, None] * x0 + np.sqrt(1 - alpha_bar[t])[:, None] * eps
    
    # Predict noise
    eps_pred = model.predict(xt, t)
    loss = np.mean((eps_pred - eps)**2)
    losses.append(loss)
    
    # Numerical gradient update
    eg = 3e-4
    for param in model.params():
        flat = param.ravel()
        n_up = min(len(flat), 120)
        inds = np.random.choice(len(flat), n_up, replace=False)
        for j in inds:
            flat[j] += eg
            lp = np.mean((model.predict(xt, t) - eps)**2)
            flat[j] -= 2*eg
            lm = np.mean((model.predict(xt, t) - eps)**2)
            flat[j] += eg
            flat[j] -= lr * (lp - lm) / (2*eg)
    
    if step % 500 == 0:
        print(f"Step {step}: loss = {loss:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.5)
plt.xlabel('Step')
plt.ylabel('MSE Loss')
plt.title('DDPM Training: Predict the Noise')
plt.grid(True, alpha=0.3)
plt.show()

print("\n🔑 Training is remarkably simple and stable:")
print("• Just MSE loss — no adversarial games, no ELBO")
print("• The target (noise ε) is always well-defined")
print("• Loss converges smoothly — compare with GAN oscillation!")

## 5. Sampling — Generating from Noise

To generate data, we reverse the process:

1. Start with $x_T \sim \mathcal{N}(0, I)$ (pure noise)
2. For $t = T, T-1, ..., 1$:
   - Predict the noise: $\hat{\epsilon} = \epsilon_\theta(x_t, t)$
   - Compute the mean: $\mu = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \hat{\epsilon}\right)$
   - Sample: $x_{t-1} = \mu + \sigma_t z$ where $z \sim \mathcal{N}(0, I)$

In [ ]:
# DDPM Sampling

def ddpm_sample(model, n_samples, T, alpha, alpha_bar, beta):
    """Generate samples by iterative denoising."""
    x = np.random.randn(n_samples, 2)  # Start from pure noise
    snapshots = {T: x.copy()}
    
    for t in reversed(range(T)):
        t_batch = np.full(n_samples, t)
        eps_pred = model.predict(x, t_batch)
        
        # Compute reverse mean
        coef1 = 1.0 / np.sqrt(alpha[t])
        coef2 = beta[t] / np.sqrt(1 - alpha_bar[t])
        mu = coef1 * (x - coef2 * eps_pred)
        
        if t > 0:
            sigma = np.sqrt(beta[t])
            x = mu + sigma * np.random.randn(n_samples, 2)
        else:
            x = mu  # No noise at final step
        
        if t in [int(0.9*T), int(0.75*T), int(0.5*T), int(0.25*T), int(0.1*T), 0]:
            snapshots[t] = x.copy()
    
    return x, snapshots

samples, snaps = ddpm_sample(model, 1500, T, alpha, alpha_bar, beta)

# Visualize the denoising trajectory
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

sorted_times = sorted(snaps.keys(), reverse=True)
for i, t in enumerate(sorted_times[:8]):
    row, col = i // 4, i % 4
    ax = axes[row, col]
    ax.scatter(snaps[t][:, 0], snaps[t][:, 1], s=2, alpha=0.3, c='coral')
    if t == 0:
        ax.scatter(data[:500, 0], data[:500, 1], s=1, alpha=0.1, c='steelblue')
    ax.set_title(f't={t}', fontsize=11)
    ax.set_xlim(-4, 5)
    ax.set_ylim(-3, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.suptitle('DDPM Sampling: Pure Noise → Data (Iterative Denoising)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Compare generated vs real
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(data[:, 0], data[:, 1], s=2, alpha=0.3, c='steelblue')
ax1.set_title('Real Data')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

ax2.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.3, c='coral')
ax2.set_title('Generated Data (DDPM)')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.suptitle('DDPM Results', fontsize=14)
plt.tight_layout()
plt.show()

print("🔑 The model generates data by iteratively predicting and removing noise.")
print("Each step makes the distribution slightly more structured.")
print("After T steps, noise becomes data!")

## 6. Three Equivalent Perspectives

The same model can be viewed through three equivalent lenses:

| Perspective | Predicts | Target |
|------------|----------|--------|
| **Noise prediction** | $\epsilon_\theta(x_t, t) \approx \epsilon$ | The noise added |
| **Data prediction** | $\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t}\epsilon_\theta}{\sqrt{\bar{\alpha}_t}}$ | The clean data |
| **Score prediction** | $s_\theta = -\frac{\epsilon_\theta}{\sqrt{1-\bar{\alpha}_t}} \approx \nabla_x \log p_t(x)$ | Direction of increasing density |

These are all the **same model**, just different ways of interpreting its output!

In [ ]:
# Three views of the same model

# Take a noisy sample and show all three predictions
t_demo = T // 3  # mid-noise level
x0_demo = data[:1]  # one data point
eps_demo = np.random.randn(1, 2)
xt_demo = np.sqrt(alpha_bar[t_demo]) * x0_demo + np.sqrt(1-alpha_bar[t_demo]) * eps_demo

# Model prediction
eps_pred = model.predict(xt_demo, np.array([t_demo]))

# Three interpretations
noise_pred = eps_pred
x0_pred = (xt_demo - np.sqrt(1-alpha_bar[t_demo]) * eps_pred) / np.sqrt(alpha_bar[t_demo])
score_pred = -eps_pred / np.sqrt(1 - alpha_bar[t_demo])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Noise prediction view
axes[0].scatter(xt_demo[0, 0], xt_demo[0, 1], c='red', s=100, zorder=5, label='Noisy xₜ')
axes[0].quiver(xt_demo[0, 0], xt_demo[0, 1], -noise_pred[0, 0], -noise_pred[0, 1],
               scale=5, color='blue', label='Remove predicted noise')
axes[0].scatter(data[:200, 0], data[:200, 1], s=1, alpha=0.1, c='gray')
axes[0].set_title(f'View 1: Noise Prediction\nε̂ = ({noise_pred[0,0]:.2f}, {noise_pred[0,1]:.2f})', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Data prediction view
axes[1].scatter(xt_demo[0, 0], xt_demo[0, 1], c='red', s=100, zorder=5, label='Noisy xₜ')
axes[1].scatter(x0_pred[0, 0], x0_pred[0, 1], c='green', s=100, zorder=5, marker='*', label='Predicted x₀')
axes[1].scatter(x0_demo[0, 0], x0_demo[0, 1], c='blue', s=100, zorder=5, marker='x', label='True x₀')
axes[1].scatter(data[:200, 0], data[:200, 1], s=1, alpha=0.1, c='gray')
axes[1].set_title(f'View 2: Data Prediction\nx̂₀ = ({x0_pred[0,0]:.2f}, {x0_pred[0,1]:.2f})', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

# Score prediction view
xx, yy = np.meshgrid(np.linspace(-3, 4, 15), np.linspace(-2, 3, 15))
grid = np.column_stack([xx.ravel(), yy.ravel()])
t_grid = np.full(len(grid), t_demo)
eps_grid = model.predict(grid, t_grid)
score_grid = -eps_grid / np.sqrt(1 - alpha_bar[t_demo])

speed = np.sqrt(score_grid[:, 0]**2 + score_grid[:, 1]**2)
axes[2].quiver(grid[:, 0], grid[:, 1], score_grid[:, 0], score_grid[:, 1],
               speed, cmap='coolwarm', alpha=0.7, scale=30)
axes[2].scatter(data[:200, 0], data[:200, 1], s=1, alpha=0.1, c='gray')
axes[2].set_title(f'View 3: Score = ∇log p(xₜ)\nArrows point toward high density', fontsize=11)
axes[2].set_aspect('equal')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Three Views of the Same Model', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("🔑 All three views are mathematically equivalent:")
print("  ε = noise that was added")
print("  x₀ = clean data estimate = (xₜ - √(1-ᾱ)ε) / √ᾱ")
print("  score = direction of increasing density = -ε / √(1-ᾱ)")

## 7. Score Matching and Langevin Dynamics

The **score function** $s(x) = \nabla_x \log p(x)$ is a vector field that points toward
regions of higher probability — like a compass that always points "uphill."

Given the score, we can sample using **Langevin dynamics**:

$$x_{t+1} = x_t + \eta \nabla_x \log p(x_t) + \sqrt{2\eta} \, z$$

- The gradient term pushes particles toward high-probability regions
- The noise term prevents particles from getting stuck
- Together: a random walk that samples from $p(x)$

In [ ]:
# Langevin dynamics with a known score function

# Known distribution: mixture of 3 Gaussians
centers = np.array([[-2, 0], [2, 0], [0, 2.5]])
sigma_mix = 0.5

def exact_score(x):
    """Score function for mixture of 3 Gaussians."""
    probs = np.zeros(len(x))
    weighted_s = np.zeros_like(x)
    for c in centers:
        p = np.exp(-0.5 * np.sum((x - c)**2, axis=1) / sigma_mix**2)
        probs += p
        weighted_s += p[:, None] * (-(x - c) / sigma_mix**2)
    return weighted_s / (probs[:, None] + 1e-10)

# Langevin dynamics
n_particles = 500
eta = 0.01  # step size
n_langevin_steps = 2000

particles = np.random.randn(n_particles, 2) * 3  # Start from diffuse noise
particle_history = [particles.copy()]

for step in range(n_langevin_steps):
    s = exact_score(particles)
    noise = np.random.randn(n_particles, 2)
    particles = particles + eta * s + np.sqrt(2 * eta) * noise
    if step in [50, 200, 500, 1999]:
        particle_history.append(particles.copy())

# Visualization
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
step_labels = ['t=0 (random)', 't=50', 't=200', 't=500', 't=2000 (converged)']

for i, (pts, label) in enumerate(zip(particle_history, step_labels)):
    ax = axes[i]
    ax.scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.4, c='coral')
    ax.scatter(centers[:, 0], centers[:, 1], c='black', s=80, marker='x', linewidths=2)
    ax.set_title(label, fontsize=10)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-3, 5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Langevin Dynamics: Random Noise → Samples from Target Distribution', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("🔑 Langevin dynamics is the precursor to diffusion models!")
print("Follow the score (gradient of log-density) + add noise → sample from p(x).")
print("Diffusion models = Langevin dynamics with a LEARNED score at multiple noise levels.")

## 8. The Complete Picture

### How Everything Connects

```
DDPM (2020)          Score Matching (2019)       Flow Matching (2023)
  │                       │                          │
  │ predict noise ε       │ predict score ∇log p     │ predict velocity v
  │                       │                          │
  └───── ε = -√(1-ᾱ) · s ─┘                          │
              │                                       │
              └──── Probability Flow ODE ──────────────┘
                         │
                    dx/dt = v(x,t)
```

### Summary of Key Ideas

| Concept | Key Insight |
|---------|-------------|
| **Forward process** | Destroying information is trivial — just add noise |
| **Reverse process** | Learning to denoise is tractable — small steps are Gaussian |
| **Training** | Just predict the noise: $\|\epsilon - \epsilon_\theta\|^2$ |
| **Sampling** | Iterative denoising: noise → data in T steps |
| **Score matching** | Predicting noise ≡ predicting score ≡ predicting clean data |
| **Noise schedule** | Controls information destruction rate; cosine > linear |
| **Flow matching** | Same models, OT paths → faster sampling (5-20 vs 1000 steps) |

### The Generative Models Landscape

| Model | Training | Sampling | Quality | Speed |
|-------|----------|----------|---------|-------|
| **GANs** (Tutorial 16) | Adversarial | 1 step | Excellent | Fast |
| **Flows** (Tutorial 17) | MLE | 1 pass | Good | Fast |
| **Flow Matching** (Tutorial 18) | Regression | 5-20 ODE steps | Excellent | Medium |
| **Diffusion** (This tutorial) | Regression | 50-1000 steps | Excellent | Slow |

Diffusion models won because:
1. **Training is dead simple** (just MSE denoising)
2. **No mode collapse** (unlike GANs)
3. **Best sample quality** (especially with guidance)
4. **Flexible architecture** (no invertibility constraint)
5. **Well-understood theory** (connects to SDEs, score matching, flows)